# 15. Model comparison

Compare the six model specifications using their validation predictions. Select the model with the highest mean validation PR-AUC, then calculate thresholds and review-capacity results. Final-test data are not used for selection.


In [ ]:
# Resolve the locked main analysis handoff bundle and freeze the Notebook 15 contract

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import math
import os

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)


COLAB_MYDRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None

if COLAB_MYDRIVE.is_dir():
    print("Google Drive is already mounted at /content/drive.")
elif drive is not None:
    try:
        drive.mount("/content/drive", force_remount=False)
    except (ValueError, OSError) as exc:
        if COLAB_MYDRIVE.is_dir():
            print("Google Drive became available after the mount warning; continuing.")
        else:
            raise RuntimeError(
                "Google Drive could not be mounted. Complete authorisation and rerun Cell 1."
            ) from exc


DEFAULT_PROJECT_ROOT = Path(
    os.environ.get("DISSERTATION_PROJECT_ROOT", "/content/drive/MyDrive/Data Final")
).expanduser()
explicit_config = os.environ.get("V2_CONFIG_PATH")
CONFIG_PATH = (
    Path(explicit_config).expanduser()
    if explicit_config
    else DEFAULT_PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
)
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Shared Version 2 config not found: {CONFIG_PATH}")

config_spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
config = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config)
REQUIRED_CONFIG_FIELDS = {
    "ANALYSIS_VERSION", "VERSION_2_MAIN_TIMING_POLICY", "V2_OUTPUT_ROOT",
    "VALIDATION_FOLDS", "FINAL_TEST_DESIGN", "TARGET_HORIZON_DAYS",
    "RANDOM_SEED", "MISSING_DATA_POLICY", "HISTORY_POLICY",
}
missing_config_fields = sorted(REQUIRED_CONFIG_FIELDS.difference(dir(config)))
if missing_config_fields:
    raise AttributeError("Shared config missing: " + ", ".join(missing_config_fields))

ANALYSIS_VERSION = str(config.ANALYSIS_VERSION)
TIMING_POLICY = dict(config.VERSION_2_MAIN_TIMING_POLICY)
VALIDATION_FOLDS = tuple(dict(item) for item in config.VALIDATION_FOLDS)
FINAL_TEST_DESIGN = dict(config.FINAL_TEST_DESIGN)
TARGET_HORIZON_DAYS = int(config.TARGET_HORIZON_DAYS)
RANDOM_SEED = int(config.RANDOM_SEED)
MISSING_DATA_POLICY = dict(config.MISSING_DATA_POLICY)
HISTORY_POLICY = dict(config.HISTORY_POLICY)
V2_OUTPUT_ROOT = Path(config.V2_OUTPUT_ROOT).expanduser()
PROJECT_ROOT = Path(getattr(config, "PROJECT_ROOT", CONFIG_PATH.parent.parent)).expanduser()
V2_ROOT = Path(getattr(config, "V2_ROOT", CONFIG_PATH.parent)).expanduser()


def quarter_end(label):
    return pd.Period(str(label), freq="Q").end_time.normalize()


def deduplicate_paths(paths):
    result = []
    for item in paths:
        item = Path(item).expanduser()
        if item not in result:
            result.append(item)
    return result


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_int64(values):
    array = np.ascontiguousarray(np.asarray(values, dtype="int64"))
    return hashlib.sha256(array.tobytes()).hexdigest()


def passed_values(series):
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "pass"})


def false_values(series):
    return series.astype(str).str.strip().str.lower().isin({"false", "0", "no"})


def is_relative_to(path, root):
    try:
        Path(path).resolve().relative_to(Path(root).resolve())
        return True
    except ValueError:
        return False


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    if pd.isna(value):
        return None
    return value


FOLDS = [f"fold{int(item['fold'])}" for item in VALIDATION_FOLDS]
TARGET = "failed_1yr"
IDENTIFIER_COLUMNS = ["report_date", "cert", "idrssd"]
HISTORY_INDICATOR = str(HISTORY_POLICY["indicator_model_name"])
DEVELOPMENT_END = quarter_end(FINAL_TEST_DESIGN["train_end"])
SELECTION_TOLERANCE = 1e-6
THRESHOLD_TOLERANCE = 1e-12
DEFAULT_DESCRIPTIVE_THRESHOLD = 0.50
CAPACITIES = (0.01, 0.05, 0.10, 0.20)
CAPACITY_LABELS = {0.01: "top_1pct", 0.05: "top_5pct", 0.10: "top_10pct", 0.20: "top_20pct"}
RUN_UTC = datetime.now(timezone.utc).isoformat()


lock_bundle_paths = [
    Path("10_Chronological_Validation_Design") / "v2_precommitted_model_selection_rule.csv",
    Path("11_EDA_and_Final_Predictor_Selection") / "locked_predictor_sets_v2.json",
    Path("11_EDA_and_Final_Predictor_Selection") / "v2_bank_only_feature_manifest.csv",
    Path("11_EDA_and_Final_Predictor_Selection") / "v2_bank_plus_macro_feature_manifest.csv",
]
model_bundle_paths = [
    Path("12_Logistic_Regression") / "cell_07_final_validation_gate.csv",
    Path("12_Logistic_Regression") / "selected_logistic_regression_configurations.json",
    Path("12_Logistic_Regression") / "selected_logistic_regression_fold_metrics.csv",
    Path("12_Logistic_Regression") / "validation_prediction_manifest.csv",
    Path("13_Random_Forest") / "cell_07_final_validation_gate.csv",
    Path("13_Random_Forest") / "selected_random_forest_configurations.json",
    Path("13_Random_Forest") / "selected_random_forest_fold_metrics.csv",
    Path("13_Random_Forest") / "validation_prediction_manifest.csv",
    Path("14_XGBoost") / "cell_07_final_validation_gate.csv",
    Path("14_XGBoost") / "selected_xgboost_configurations.json",
    Path("14_XGBoost") / "selected_xgboost_fold_metrics.csv",
    Path("14_XGBoost") / "validation_prediction_manifest.csv",
]


def bundle_complete(root, relative_paths):
    return all((root / relative).is_file() for relative in relative_paths)


candidate_roots = deduplicate_paths([
    V2_OUTPUT_ROOT / ANALYSIS_VERSION,
    V2_OUTPUT_ROOT,
    V2_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    V2_ROOT / "Notebook Outputs",
    PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    PROJECT_ROOT / "Notebook Outputs",
    DEFAULT_PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    DEFAULT_PROJECT_ROOT / "Notebook Outputs",
])
lock_roots = [root for root in candidate_roots if bundle_complete(root, lock_bundle_paths)]
model_roots = [root for root in candidate_roots if bundle_complete(root, model_bundle_paths)]
if not lock_roots:
    diagnostic_lines = []
    for root in candidate_roots:
        missing = [str(item) for item in lock_bundle_paths if not (root / item).is_file()]
        diagnostic_lines.append(f"{root}\n  missing: " + "; ".join(missing))
    raise FileNotFoundError(
        "No complete Version 2 Notebook 10–11 lock bundle was found.\n"
        + "\n".join(diagnostic_lines)
    )
if not model_roots:
    diagnostic_lines = []
    for root in candidate_roots:
        missing = [str(item) for item in model_bundle_paths if not (root / item).is_file()]
        diagnostic_lines.append(f"{root}\n  missing: " + "; ".join(missing))
    raise FileNotFoundError(
        "No complete Version 2 Notebook 12–14 model bundle was found.\n"
        + "\n".join(diagnostic_lines)
    )

LOCK_OUTPUT_ROOT = lock_roots[0]
MODEL_OUTPUT_ROOT = model_roots[0]
NB15_HANDOFF_ROOT = (
    MODEL_OUTPUT_ROOT
    if is_relative_to(MODEL_OUTPUT_ROOT, V2_ROOT)
    else V2_OUTPUT_ROOT / ANALYSIS_VERSION
)
NB15_OUTPUT_DIR = NB15_HANDOFF_ROOT / "15_Validation_Comparison_and_Model_Lock"
NB15_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SELECTION_RULE_PATH = (
    LOCK_OUTPUT_ROOT
    / "10_Chronological_Validation_Design"
    / "v2_precommitted_model_selection_rule.csv"
)
LOCKED_SETS_PATH = (
    LOCK_OUTPUT_ROOT
    / "11_EDA_and_Final_Predictor_Selection"
    / "locked_predictor_sets_v2.json"
)
BANK_MANIFEST_PATH = (
    LOCK_OUTPUT_ROOT
    / "11_EDA_and_Final_Predictor_Selection"
    / "v2_bank_only_feature_manifest.csv"
)
FULL_MANIFEST_PATH = (
    LOCK_OUTPUT_ROOT
    / "11_EDA_and_Final_Predictor_Selection"
    / "v2_bank_plus_macro_feature_manifest.csv"
)

setup_checks = {
    "Analysis is the Version 2 lagged-bank main design":
        ANALYSIS_VERSION == "v2_bank_lag1_main",
    "Exact t-1 bank and macro timing remains locked":
        TIMING_POLICY.get("bank_predictor_source") == "exact_t_minus_1"
        and TIMING_POLICY.get("macro_predictor_source") == "t_minus_1",
    "Target remains anchored at t over 365 days":
        TIMING_POLICY.get("target_window") == "(t, t_plus_365_days]"
        and TARGET_HORIZON_DAYS == 365,
    "Exactly three validation folds are available":
        FOLDS == ["fold1", "fold2", "fold3"],
    "All required Notebook 10–14 handoffs are present":
        bundle_complete(LOCK_OUTPUT_ROOT, lock_bundle_paths)
        and bundle_complete(MODEL_OUTPUT_ROOT, model_bundle_paths),
    "Notebook 15 writes only inside the V2 root":
        is_relative_to(NB15_OUTPUT_DIR, V2_ROOT),
    "Four quarterly capacity groups are predeclared":
        CAPACITIES == (0.01, 0.05, 0.10, 0.20),
    "No final-test path is part of the Notebook 15 input bundle":
        all(
            "test" not in str(item).lower()
            for item in [*lock_bundle_paths, *model_bundle_paths]
        ),
}
notebook_15_cell_01_gate = pd.DataFrame({
    "check": setup_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in setup_checks.values()],
    "passed": setup_checks.values(),
})
notebook_15_cell_01_gate.to_csv(NB15_OUTPUT_DIR / "cell_01_setup_gate.csv", index=False)

notebook_contract = pd.DataFrame({
    "item": [
        "selection universe", "primary selection metric", "first tie-break",
        "second tie-break", "selection tolerance", "threshold source",
        "threshold objective", "fold thresholds", "capacity groups",
        "quarterly group size", "capacity ranking", "primary capacity result",
        "secondary capacity result", "capacity role in selection",
        "final-test information", "Version 1 threshold",
    ],
    "locked_rule": [
        "six V2 lagged-bank specifications selected within Notebooks 12–14",
        "highest unweighted mean of three fold raw PR-AUC values",
        "higher unweighted mean ROC-AUC within 1e-6 of best mean PR-AUC",
        "lower sample standard deviation of fold PR-AUC within 1e-6 again",
        SELECTION_TOLERANCE,
        "pooled selected-model out-of-time validation predictions only",
        "maximum pooled F1; tied values use higher recall, higher precision, then higher threshold",
        "stability diagnostic only; never handed downstream",
        "1%, 5%, 10% and 20% within each validation quarter",
        "max(1, ceil(capacity multiplied by active bank-quarter rows))",
        "descending probability; CERT, IDRSSD and panel_row_id ascending tie-breaks",
        "distinct failed banks captured in at least one positive pre-failure quarter",
        "positive bank-quarter capture",
        "descriptive only; never affects model selection",
        "prohibited",
        "not loaded, compared or reused",
    ],
})
notebook_contract.to_csv(NB15_OUTPUT_DIR / "cell_01_notebook_15_contract.csv", index=False)

print("Notebook 15 Cell 1 validation gate:")
display(notebook_15_cell_01_gate)
print(f"\nNotebook 10–11 lock bundle: {LOCK_OUTPUT_ROOT}")
print(f"Notebook 12–14 model bundle: {MODEL_OUTPUT_ROOT}")
print(f"Notebook 15 output directory: {NB15_OUTPUT_DIR}")
print("\nNotebook 15 locked contract:")
with pd.option_context("display.max_colwidth", None):
    display(notebook_contract)
failed = [name for name, passed in setup_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 1 blocked: " + "; ".join(failed))


In [ ]:
# Load and audit the six selected validation handoffs without touching final-test data

if not notebook_15_cell_01_gate["passed"].all():
    raise AssertionError("Cell 2 blocked because Cell 1 did not pass.")

with open(LOCKED_SETS_PATH, "r", encoding="utf-8") as handle:
    locked_sets = json.load(handle)
bank_manifest = pd.read_csv(BANK_MANIFEST_PATH).sort_values("feature_order")
full_manifest = pd.read_csv(FULL_MANIFEST_PATH).sort_values("feature_order")
model_selection_rule = pd.read_csv(MODEL_SELECTION_RULE_PATH)

BANK_ONLY_PREDICTORS = bank_manifest["version_2_feature"].astype(str).tolist()
BANK_PLUS_MACRO_PREDICTORS = full_manifest["version_2_feature"].astype(str).tolist()
SPECIFICATION_FEATURES = {
    "bank_only": BANK_ONLY_PREDICTORS,
    "bank_plus_macro": BANK_PLUS_MACRO_PREDICTORS,
}

MODEL_INPUTS = {
    "logistic_regression": {
        "display_name": "Logistic Regression",
        "directory": MODEL_OUTPUT_ROOT / "12_Logistic_Regression",
        "selected_csv": "selected_logistic_regression_configurations.csv",
        "selected_json": "selected_logistic_regression_configurations.json",
        "fold_metrics": "selected_logistic_regression_fold_metrics.csv",
        "prediction_manifest": "validation_prediction_manifest.csv",
        "preprocessing_manifest": "preprocessing_parameter_manifest.csv",
        "tuning_contract": "cell_01_logistic_tuning_contract.csv",
        "final_gate": "cell_07_final_validation_gate.csv",
        "hyperparameter_columns": [
            "regularisation_penalty", "solver", "C", "positive_weight_multiplier",
        ],
    },
    "random_forest": {
        "display_name": "Random Forest",
        "directory": MODEL_OUTPUT_ROOT / "13_Random_Forest",
        "selected_csv": "selected_random_forest_configurations.csv",
        "selected_json": "selected_random_forest_configurations.json",
        "fold_metrics": "selected_random_forest_fold_metrics.csv",
        "prediction_manifest": "validation_prediction_manifest.csv",
        "preprocessing_manifest": "preprocessing_parameter_manifest.csv",
        "tuning_contract": "cell_01_random_forest_tuning_contract.csv",
        "final_gate": "cell_07_final_validation_gate.csv",
        "hyperparameter_columns": [
            "n_estimators", "max_depth", "min_samples_leaf", "max_features",
            "positive_weight_multiplier", "max_samples",
        ],
    },
    "xgboost": {
        "display_name": "XGBoost",
        "directory": MODEL_OUTPUT_ROOT / "14_XGBoost",
        "selected_csv": "selected_xgboost_configurations.csv",
        "selected_json": "selected_xgboost_configurations.json",
        "fold_metrics": "selected_xgboost_fold_metrics.csv",
        "prediction_manifest": "validation_prediction_manifest.csv",
        "preprocessing_manifest": "preprocessing_parameter_manifest.csv",
        "tuning_contract": "cell_01_xgboost_tuning_contract.csv",
        "final_gate": "cell_07_final_validation_gate.csv",
        "hyperparameter_columns": [
            "n_estimators", "max_depth", "learning_rate", "min_child_weight",
            "subsample", "colsample_bytree", "gamma", "reg_lambda", "reg_alpha",
            "positive_weight_multiplier", "tree_method",
        ],
    },
}

selected_records = {}
selected_fold_metrics_by_family = {}
prediction_sources = []
preprocessing_audit_rows = []
handoff_audit_rows = []
loaded_input_paths = [
    CONFIG_PATH, MODEL_SELECTION_RULE_PATH, LOCKED_SETS_PATH,
    BANK_MANIFEST_PATH, FULL_MANIFEST_PATH,
]

for family, contract in MODEL_INPUTS.items():
    directory = contract["directory"]
    selected_csv_path = directory / contract["selected_csv"]
    selected_json_path = directory / contract["selected_json"]
    fold_metrics_path = directory / contract["fold_metrics"]
    prediction_manifest_path = directory / contract["prediction_manifest"]
    preprocessing_manifest_path = directory / contract["preprocessing_manifest"]
    tuning_contract_path = directory / contract["tuning_contract"]
    final_gate_path = directory / contract["final_gate"]
    family_paths = [
        selected_csv_path, selected_json_path, fold_metrics_path,
        prediction_manifest_path, preprocessing_manifest_path,
        tuning_contract_path, final_gate_path,
    ]
    missing_family_paths = [path for path in family_paths if not path.is_file()]
    if missing_family_paths:
        raise FileNotFoundError(
            f"Missing {family} handoff files: " + "; ".join(map(str, missing_family_paths))
        )
    loaded_input_paths.extend(family_paths)

    selected_csv = pd.read_csv(selected_csv_path)
    with open(selected_json_path, "r", encoding="utf-8") as handle:
        selected_json = json.load(handle)
    fold_metrics = pd.read_csv(fold_metrics_path)
    prediction_manifest = pd.read_csv(prediction_manifest_path)
    preprocessing_manifest = pd.read_csv(preprocessing_manifest_path)
    tuning_contract = pd.read_csv(tuning_contract_path)
    final_gate = pd.read_csv(final_gate_path)

    if not isinstance(selected_json, list):
        raise TypeError(f"{selected_json_path.name} must contain a JSON list.")
    json_frame = pd.DataFrame(selected_json)
    csv_keys = selected_csv[["specification", "config_id"]].astype(str).sort_values(
        ["specification", "config_id"]
    ).reset_index(drop=True)
    json_keys = json_frame[["specification", "config_id"]].astype(str).sort_values(
        ["specification", "config_id"]
    ).reset_index(drop=True)

    family_checks = {
        "final_gate_pass": passed_values(final_gate["passed"]).all(),
        "two_selected_specifications":
            len(selected_csv) == len(selected_json) == 2
            and set(selected_csv["specification"]) == {"bank_only", "bank_plus_macro"},
        "csv_json_selected_keys_match": csv_keys.equals(json_keys),
        "three_fold_rows_per_selected_specification":
            len(fold_metrics) == 6
            and fold_metrics.groupby("specification")["fold"].nunique().eq(3).all(),
        "threshold_selection_was_deferred":
            false_values(selected_csv["threshold_selected"]).all(),
        "final_test_was_not_used":
            false_values(selected_csv["final_test_used"]).all(),
        "version_1_winner_was_not_reused":
            false_values(selected_csv["version_1_winner_reused"]).all(),
        "six_training_fold_preprocessing_files":
            len(preprocessing_manifest) == 6
            and preprocessing_manifest.groupby("specification")["fold"].nunique().eq(3).all(),
    }
    handoff_audit_rows.extend({
        "model_family": family,
        "check": check,
        "passed": bool(value),
    } for check, value in family_checks.items())

    selected_fold_metrics_by_family[family] = fold_metrics.copy()
    for record in selected_json:
        specification = str(record["specification"])
        model_id = f"{family}__{specification}"
        selected_records[model_id] = to_builtin(record)
        config_id = str(record["config_id"])
        matched_predictions = prediction_manifest.loc[
            prediction_manifest["specification"].astype(str).eq(specification)
            & prediction_manifest["config_id"].astype(str).eq(config_id)
        ].copy()
        if set(matched_predictions["fold"].astype(str)) != set(FOLDS):
            raise AssertionError(
                f"Selected prediction manifest is incomplete for {model_id} / {config_id}."
            )
        for row in matched_predictions.itertuples(index=False):
            prediction_path = directory / str(row.file)
            if not prediction_path.is_file():
                raise FileNotFoundError(f"Selected validation prediction missing: {prediction_path}")
            loaded_input_paths.append(prediction_path)
            prediction_sources.append({
                "model_family": family,
                "model_family_display": contract["display_name"],
                "specification": specification,
                "model_id": model_id,
                "fold": str(row.fold),
                "config_id": config_id,
                "file": str(prediction_path),
                "expected_observations": int(row.observations),
                "expected_positive_bank_quarters": int(row.positive_bank_quarters),
                "expected_panel_row_id_sha256": str(row.panel_row_id_sha256),
                "expected_file_sha256": str(row.sha256),
            })

    for row in preprocessing_manifest.itertuples(index=False):
        preprocessing_path = directory / str(row.file)
        if not preprocessing_path.is_file():
            raise FileNotFoundError(f"Preprocessing file missing: {preprocessing_path}")
        loaded_input_paths.append(preprocessing_path)
        parameters = json.loads(preprocessing_path.read_text(encoding="utf-8"))
        preprocessing_audit_rows.append({
            "model_family": family,
            "specification": str(row.specification),
            "fold": str(row.fold),
            "file": str(preprocessing_path),
            "file_sha256_matches_manifest": sha256_file(preprocessing_path) == str(row.sha256),
            "predictor_order_matches_lock":
                parameters["predictors"] == SPECIFICATION_FEATURES[str(row.specification)],
            "validation_parameters_fitted": bool(parameters["validation_parameters_fitted"]),
            "test_data_used": bool(parameters["test_data_used"]),
        })

handoff_audit = pd.DataFrame(handoff_audit_rows)
prediction_source_manifest = pd.DataFrame(prediction_sources).sort_values(
    ["model_family", "specification", "fold"]
).reset_index(drop=True)
preprocessing_handoff_audit = pd.DataFrame(preprocessing_audit_rows)
loaded_input_paths = deduplicate_paths(loaded_input_paths)
input_hashes_before = {str(path): sha256_file(path) for path in loaded_input_paths}

input_checks = {
    "All 24 model-family handoff checks pass":
        len(handoff_audit) == 24 and handoff_audit["passed"].all(),
    "Exactly six selected model specifications are present":
        len(selected_records) == 6,
    "Exactly 18 selected fold prediction files are resolved":
        len(prediction_source_manifest) == 18
        and prediction_source_manifest["file"].nunique() == 18,
    "All 18 preprocessing records are training-only and match the feature locks":
        len(preprocessing_handoff_audit) == 18
        and preprocessing_handoff_audit[
            ["file_sha256_matches_manifest", "predictor_order_matches_lock"]
        ].all().all()
        and not preprocessing_handoff_audit["validation_parameters_fitted"].any()
        and not preprocessing_handoff_audit["test_data_used"].any(),
    "Bank-only predictor lock contains 19 ordered features":
        len(BANK_ONLY_PREDICTORS) == 19
        and BANK_ONLY_PREDICTORS == locked_sets["bank_only_features"],
    "Bank-plus-macro predictor lock contains 23 ordered features":
        len(BANK_PLUS_MACRO_PREDICTORS) == 23
        and BANK_PLUS_MACRO_PREDICTORS == locked_sets["bank_plus_macro_features"],
    "History indicator is included in both predictor locks":
        HISTORY_INDICATOR == "insufficient_history_5q_lag1"
        and all(HISTORY_INDICATOR in values for values in SPECIFICATION_FEATURES.values()),
    "Notebook 10 selection rule is validation-only":
        len(model_selection_rule) == 3
        and not model_selection_rule["uses_final_test"].astype(str).str.strip().str.lower().isin(
            {"true", "1", "yes"}
        ).any()
        and model_selection_rule.iloc[0]["rule"].startswith("Highest unweighted mean PR-AUC"),
    "No loaded input path is a final-test or Version 1 artifact":
        all("final_test" not in str(path).lower() for path in loaded_input_paths)
        and all("version 1" not in str(path).lower() for path in loaded_input_paths),
}
notebook_15_cell_02_gate = pd.DataFrame({
    "check": input_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in input_checks.values()],
    "passed": input_checks.values(),
})

handoff_audit.to_csv(NB15_OUTPUT_DIR / "cell_02_model_handoff_audit.csv", index=False)
prediction_source_manifest.to_csv(
    NB15_OUTPUT_DIR / "cell_02_selected_prediction_source_manifest.csv", index=False
)
preprocessing_handoff_audit.to_csv(
    NB15_OUTPUT_DIR / "cell_02_preprocessing_handoff_audit.csv", index=False
)
pd.DataFrame([
    {"input_file": path, "sha256_before": digest}
    for path, digest in input_hashes_before.items()
]).to_csv(NB15_OUTPUT_DIR / "cell_02_input_hashes.csv", index=False)
notebook_15_cell_02_gate.to_csv(
    NB15_OUTPUT_DIR / "cell_02_handoff_validation_gate.csv", index=False
)

print("Notebook 15 Cell 2 validation gate:")
display(notebook_15_cell_02_gate)
print("\nSelected prediction sources:")
display(prediction_source_manifest[
    ["model_family_display", "specification", "fold", "config_id", "expected_observations"]
])
failed = [name for name, passed in input_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 2 blocked: " + "; ".join(failed))


In [ ]:
# Recompute fold metrics and select the validation leader using raw PR-AUC

if not notebook_15_cell_02_gate["passed"].all():
    raise AssertionError("Cell 3 blocked because Cell 2 did not pass.")


def probability_metrics(y_true, probability, threshold=DEFAULT_DESCRIPTIVE_THRESHOLD):
    y_true = np.asarray(y_true, dtype="int8")
    probability = np.asarray(probability, dtype="float64")
    predicted = (probability >= float(threshold)).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    prevalence = float(np.mean(y_true))
    pr_auc = float(average_precision_score(y_true, probability))
    return {
        "observations": int(len(y_true)),
        "positive_bank_quarters": int(y_true.sum()),
        "observed_prevalence": prevalence,
        "pr_auc": pr_auc,
        "pr_auc_divided_by_prevalence": float(pr_auc / prevalence),
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "threshold": float(threshold),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    }


def choose_validation_leader(comparison, tolerance=SELECTION_TOLERANCE):
    stage_1 = comparison.loc[
        comparison["mean_pr_auc"] >= comparison["mean_pr_auc"].max() - tolerance
    ].copy()
    stage_2 = stage_1.loc[
        stage_1["mean_roc_auc"] >= stage_1["mean_roc_auc"].max() - tolerance
    ].copy()
    minimum_std = stage_2["std_pr_auc"].min()
    stage_3 = stage_2.loc[
        stage_2["std_pr_auc"] <= minimum_std + tolerance
    ].copy()
    if len(stage_3) != 1:
        raise AssertionError(
            "The precommitted PR-AUC, ROC-AUC and PR-AUC-variability rules did not "
            "produce a unique leader: " + ", ".join(stage_3["model_id"].astype(str))
        )
    return stage_3.iloc[0], stage_1, stage_2


predictions_by_model = {}
prediction_audit_rows = []
fold_metric_rows = []

for source in prediction_source_manifest.itertuples(index=False):
    path = Path(source.file)
    frame = pd.read_parquet(path)
    required_columns = {
        "panel_row_id", *IDENTIFIER_COLUMNS, TARGET, "analysis_version",
        "specification", "fold", "config_id", "predicted_probability",
    }
    missing_columns = sorted(required_columns.difference(frame.columns))
    if missing_columns:
        raise KeyError(f"{path.name} missing columns: {missing_columns}")
    frame = frame.copy()
    frame["report_date"] = pd.to_datetime(frame["report_date"]).dt.normalize()
    frame = frame.sort_values("panel_row_id", kind="mergesort").reset_index(drop=True)
    probability = frame["predicted_probability"].to_numpy(dtype="float64")
    y_true = frame[TARGET].to_numpy(dtype="int8")
    audit = {
        "model_id": source.model_id,
        "model_family": source.model_family,
        "specification": source.specification,
        "fold": source.fold,
        "config_id": source.config_id,
        "file": source.file,
        "file_sha256_matches_manifest": sha256_file(path) == source.expected_file_sha256,
        "observations": int(len(frame)),
        "expected_observations": int(source.expected_observations),
        "positive_bank_quarters": int(y_true.sum()),
        "expected_positive_bank_quarters": int(source.expected_positive_bank_quarters),
        "panel_row_id_sha256": sha256_int64(frame["panel_row_id"]),
        "expected_panel_row_id_sha256": source.expected_panel_row_id_sha256,
        "duplicate_panel_row_ids": int(frame["panel_row_id"].duplicated().sum()),
        "minimum_reference_date": frame["report_date"].min(),
        "maximum_reference_date": frame["report_date"].max(),
        "probability_min": float(probability.min()),
        "probability_max": float(probability.max()),
        "probability_finite": bool(np.isfinite(probability).all()),
        "identifiers_complete": bool(frame[IDENTIFIER_COLUMNS].notna().all().all()),
        "analysis_version_matches": frame["analysis_version"].astype(str).eq(ANALYSIS_VERSION).all(),
        "specification_matches": frame["specification"].astype(str).eq(source.specification).all(),
        "fold_matches": frame["fold"].astype(str).eq(source.fold).all(),
        "config_id_matches": frame["config_id"].astype(str).eq(source.config_id).all(),
    }
    prediction_audit_rows.append(audit)
    predictions_by_model.setdefault(source.model_id, []).append(frame)

    recalculated = probability_metrics(y_true, probability, DEFAULT_DESCRIPTIVE_THRESHOLD)
    saved = selected_fold_metrics_by_family[source.model_family].loc[
        selected_fold_metrics_by_family[source.model_family]["specification"].astype(str).eq(
            source.specification
        )
        & selected_fold_metrics_by_family[source.model_family]["fold"].astype(str).eq(
            source.fold
        )
        & selected_fold_metrics_by_family[source.model_family]["config_id"].astype(str).eq(
            source.config_id
        )
    ]
    if len(saved) != 1:
        raise AssertionError(f"Saved fold metric row is not unique for {source.model_id}/{source.fold}.")
    saved = saved.iloc[0]
    fold_metric_rows.append({
        "model_id": source.model_id,
        "model_family": source.model_family,
        "model_family_display": source.model_family_display,
        "specification": source.specification,
        "fold": source.fold,
        "config_id": source.config_id,
        **{f"recalculated_{key}": value for key, value in recalculated.items()},
        "saved_pr_auc": float(saved["pr_auc"]),
        "saved_roc_auc": float(saved["roc_auc"]),
        "saved_observed_prevalence": float(saved["observed_prevalence"]),
        "saved_recall_at_0p5": float(saved["recall_at_0p5"]),
        "saved_f1_at_0p5": float(saved["f1_at_0p5"]),
        "all_saved_metrics_match": bool(
            np.isclose(recalculated["pr_auc"], saved["pr_auc"], rtol=0.0, atol=1e-12)
            and np.isclose(recalculated["roc_auc"], saved["roc_auc"], rtol=0.0, atol=1e-12)
            and np.isclose(
                recalculated["observed_prevalence"], saved["observed_prevalence"],
                rtol=0.0, atol=1e-12,
            )
            and np.isclose(recalculated["recall"], saved["recall_at_0p5"], rtol=0.0, atol=1e-12)
            and np.isclose(recalculated["f1"], saved["f1_at_0p5"], rtol=0.0, atol=1e-12)
        ),
    })

prediction_audit = pd.DataFrame(prediction_audit_rows)
fold_comparison = pd.DataFrame(fold_metric_rows).sort_values(
    ["model_family", "specification", "fold"]
).reset_index(drop=True)

for model_id, frames in predictions_by_model.items():
    pooled = pd.concat(frames, ignore_index=True)
    if pooled["panel_row_id"].duplicated().any():
        raise AssertionError(f"Validation folds overlap for {model_id}.")
    predictions_by_model[model_id] = pooled.sort_values(
        ["report_date", "panel_row_id"], kind="mergesort"
    ).reset_index(drop=True)

# All six specifications must score the identical validation rows and targets.
cross_model_rows = []
reference_model_id = sorted(predictions_by_model)[0]
reference = predictions_by_model[reference_model_id].sort_values(
    "panel_row_id", kind="mergesort"
).reset_index(drop=True)
for model_id, frame in sorted(predictions_by_model.items()):
    candidate = frame.sort_values("panel_row_id", kind="mergesort").reset_index(drop=True)
    cross_model_rows.append({
        "reference_model_id": reference_model_id,
        "candidate_model_id": model_id,
        "observations_match": len(candidate) == len(reference),
        "panel_row_ids_match": candidate["panel_row_id"].equals(reference["panel_row_id"]),
        "targets_match": candidate[TARGET].equals(reference[TARGET]),
        "dates_match": candidate["report_date"].equals(reference["report_date"]),
        "certs_match": candidate["cert"].equals(reference["cert"]),
        "idrssd_match": candidate["idrssd"].equals(reference["idrssd"]),
    })
cross_model_reconciliation = pd.DataFrame(cross_model_rows)

mean_comparison = (
    fold_comparison.groupby(
        ["model_id", "model_family", "model_family_display", "specification", "config_id"],
        as_index=False,
    )
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("recalculated_pr_auc", "mean"),
        std_pr_auc=("recalculated_pr_auc", "std"),
        min_pr_auc=("recalculated_pr_auc", "min"),
        max_pr_auc=("recalculated_pr_auc", "max"),
        mean_roc_auc=("recalculated_roc_auc", "mean"),
        std_roc_auc=("recalculated_roc_auc", "std"),
        mean_prevalence=("recalculated_observed_prevalence", "mean"),
        mean_pr_auc_lift=("recalculated_pr_auc_divided_by_prevalence", "mean"),
        mean_recall_at_0p5=("recalculated_recall", "mean"),
        mean_precision_at_0p5=("recalculated_precision", "mean"),
        mean_f1_at_0p5=("recalculated_f1", "mean"),
    )
)
leader, pr_auc_tie_set, roc_auc_tie_set = choose_validation_leader(mean_comparison)
LEADER_MODEL_ID = str(leader["model_id"])
mean_comparison["selection_status"] = np.where(
    mean_comparison["model_id"].eq(LEADER_MODEL_ID), "LOCKED VALIDATION LEADER", "COMPARATOR"
)
mean_comparison = mean_comparison.sort_values(
    ["mean_pr_auc", "mean_roc_auc", "std_pr_auc"],
    ascending=[False, False, True],
    kind="mergesort",
).reset_index(drop=True)
mean_comparison.insert(0, "validation_rank", np.arange(1, len(mean_comparison) + 1))

metric_checks = {
    "All 18 selected prediction files pass their manifest and content audit":
        len(prediction_audit) == 18
        and prediction_audit[
            [
                "file_sha256_matches_manifest", "probability_finite", "identifiers_complete",
                "analysis_version_matches", "specification_matches", "fold_matches",
                "config_id_matches",
            ]
        ].all().all()
        and prediction_audit["observations"].eq(prediction_audit["expected_observations"]).all()
        and prediction_audit["positive_bank_quarters"].eq(
            prediction_audit["expected_positive_bank_quarters"]
        ).all()
        and prediction_audit["panel_row_id_sha256"].eq(
            prediction_audit["expected_panel_row_id_sha256"]
        ).all()
        and prediction_audit["duplicate_panel_row_ids"].eq(0).all()
        and prediction_audit["probability_min"].ge(0).all()
        and prediction_audit["probability_max"].le(1).all()
        and pd.to_datetime(prediction_audit["maximum_reference_date"]).le(DEVELOPMENT_END).all(),
    "Every saved fold metric independently recomputes":
        len(fold_comparison) == 18 and fold_comparison["all_saved_metrics_match"].all(),
    "Every fold reports prevalence, raw PR-AUC and PR-AUC lift":
        fold_comparison[
            [
                "recalculated_observed_prevalence", "recalculated_pr_auc",
                "recalculated_pr_auc_divided_by_prevalence",
            ]
        ].notna().all().all()
        and np.isfinite(fold_comparison[
            [
                "recalculated_observed_prevalence", "recalculated_pr_auc",
                "recalculated_pr_auc_divided_by_prevalence",
            ]
        ].to_numpy()).all(),
    "All six models use identical validation rows and targets":
        len(cross_model_reconciliation) == 6
        and cross_model_reconciliation.drop(columns=["reference_model_id", "candidate_model_id"]).all().all(),
    "All six comparisons use exactly three unweighted folds":
        len(mean_comparison) == 6 and mean_comparison["folds"].eq(3).all(),
    "The precommitted hierarchy produces exactly one validation leader":
        mean_comparison["selection_status"].eq("LOCKED VALIDATION LEADER").sum() == 1,
}
notebook_15_cell_03_gate = pd.DataFrame({
    "check": metric_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in metric_checks.values()],
    "passed": metric_checks.values(),
})

prediction_audit.to_csv(NB15_OUTPUT_DIR / "cell_03_selected_prediction_audit.csv", index=False)
cross_model_reconciliation.to_csv(
    NB15_OUTPUT_DIR / "cell_03_cross_model_row_reconciliation.csv", index=False
)
fold_comparison.to_csv(
    NB15_OUTPUT_DIR / "v2_six_specification_fold_validation_comparison.csv", index=False
)
mean_comparison.to_csv(
    NB15_OUTPUT_DIR / "v2_six_specification_mean_validation_comparison.csv", index=False
)
notebook_15_cell_03_gate.to_csv(
    NB15_OUTPUT_DIR / "cell_03_metric_recomputation_and_selection_gate.csv", index=False
)

print("Notebook 15 Cell 3 validation gate:")
display(notebook_15_cell_03_gate)
print("\nFold-level comparison (raw validation metrics; threshold metrics at 0.50 are descriptive):")
display(fold_comparison[[
    "model_family_display", "specification", "fold", "config_id",
    "recalculated_observed_prevalence", "recalculated_pr_auc",
    "recalculated_pr_auc_divided_by_prevalence", "recalculated_roc_auc",
    "recalculated_recall", "recalculated_f1",
]])
print("\nSix-specification unweighted mean comparison:")
display(mean_comparison[[
    "validation_rank", "model_family_display", "specification", "config_id",
    "mean_pr_auc", "std_pr_auc", "mean_roc_auc", "mean_recall_at_0p5",
    "mean_f1_at_0p5", "selection_status",
]])
print(f"\nValidation leader: {LEADER_MODEL_ID}")
failed = [name for name, passed in metric_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 3 blocked: " + "; ".join(failed))


In [ ]:
# Select six new pooled-validation F1 thresholds and assess fold stability

if not notebook_15_cell_03_gate["passed"].all():
    raise AssertionError("Cell 4 blocked because Cell 3 did not pass.")


def f1_optimal_threshold(y_true, probability, tolerance=THRESHOLD_TOLERANCE):
    y_true = np.asarray(y_true, dtype="int8")
    probability = np.asarray(probability, dtype="float64")
    precision, recall, thresholds = precision_recall_curve(y_true, probability)
    if len(thresholds) == 0:
        raise AssertionError("No finite threshold candidate was produced.")
    precision = precision[:-1]
    recall = recall[:-1]
    f1_values = np.divide(
        2.0 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision, dtype="float64"),
        where=(precision + recall) > 0,
    )
    candidates = pd.DataFrame({
        "threshold": thresholds.astype("float64"),
        "precision": precision.astype("float64"),
        "recall": recall.astype("float64"),
        "f1": f1_values.astype("float64"),
    })
    best_f1 = candidates["f1"].max()
    tied = candidates.loc[candidates["f1"] >= best_f1 - tolerance].copy()
    selected = tied.sort_values(
        ["recall", "precision", "threshold"],
        ascending=[False, False, False],
        kind="mergesort",
    ).iloc[0]
    return {
        "threshold": float(selected["threshold"]),
        "precision": float(selected["precision"]),
        "recall": float(selected["recall"]),
        "f1": float(selected["f1"]),
        "candidate_thresholds": int(len(candidates)),
        "maximum_f1_tied_candidates": int(len(tied)),
    }


threshold_rows = []
fold_threshold_rows = []
locked_threshold_performance_rows = []

for model_id, frame in sorted(predictions_by_model.items()):
    family, specification = model_id.split("__", 1)
    pooled_result = f1_optimal_threshold(frame[TARGET], frame["predicted_probability"])
    threshold_rows.append({
        "model_id": model_id,
        "model_family": family,
        "model_family_display": MODEL_INPUTS[family]["display_name"],
        "specification": specification,
        "config_id": str(selected_records[model_id]["config_id"]),
        "threshold_source": "pooled out-of-time validation predictions from folds 1–3",
        "threshold_selection_metric": "maximum F1",
        "threshold_tie_breaks": "higher recall; higher precision; higher threshold",
        "fold_specific_thresholds_used_downstream": False,
        "version_1_threshold_reused": False,
        **pooled_result,
    })
    pooled_threshold = pooled_result["threshold"]

    for fold in FOLDS:
        fold_frame = frame.loc[frame["fold"].astype(str).eq(fold)].copy()
        fold_result = f1_optimal_threshold(
            fold_frame[TARGET], fold_frame["predicted_probability"]
        )
        fold_threshold_rows.append({
            "model_id": model_id,
            "model_family": family,
            "specification": specification,
            "fold": fold,
            "diagnostic_only": True,
            "pooled_locked_threshold": pooled_threshold,
            "fold_specific_threshold": fold_result["threshold"],
            "fold_minus_pooled_threshold": fold_result["threshold"] - pooled_threshold,
            "fold_specific_precision": fold_result["precision"],
            "fold_specific_recall": fold_result["recall"],
            "fold_specific_f1": fold_result["f1"],
            "candidate_thresholds": fold_result["candidate_thresholds"],
        })

    for scope in [*FOLDS, "pooled_oot_validation"]:
        scope_frame = frame if scope == "pooled_oot_validation" else frame.loc[
            frame["fold"].astype(str).eq(scope)
        ]
        metrics = probability_metrics(
            scope_frame[TARGET], scope_frame["predicted_probability"], pooled_threshold
        )
        locked_threshold_performance_rows.append({
            "model_id": model_id,
            "model_family": family,
            "model_family_display": MODEL_INPUTS[family]["display_name"],
            "specification": specification,
            "scope": scope,
            "operating_threshold": pooled_threshold,
            "threshold_source": "pooled_oot_validation",
            **metrics,
        })

pooled_thresholds = pd.DataFrame(threshold_rows).sort_values(
    ["model_family", "specification"]
).reset_index(drop=True)
fold_threshold_stability = pd.DataFrame(fold_threshold_rows).sort_values(
    ["model_family", "specification", "fold"]
).reset_index(drop=True)
locked_threshold_performance = pd.DataFrame(locked_threshold_performance_rows).sort_values(
    ["model_family", "specification", "scope"]
).reset_index(drop=True)

threshold_stability_summary = (
    fold_threshold_stability.groupby(
        ["model_id", "model_family", "specification", "pooled_locked_threshold"],
        as_index=False,
    )
    .agg(
        fold_threshold_min=("fold_specific_threshold", "min"),
        fold_threshold_max=("fold_specific_threshold", "max"),
        fold_threshold_mean=("fold_specific_threshold", "mean"),
        fold_threshold_std=("fold_specific_threshold", "std"),
        maximum_absolute_fold_minus_pooled=("fold_minus_pooled_threshold", lambda values: values.abs().max()),
    )
)

pooled_operating_metrics = locked_threshold_performance.loc[
    locked_threshold_performance["scope"].eq("pooled_oot_validation"),
    ["model_id", "operating_threshold", "precision", "recall", "f1", "true_positives", "false_positives"],
].rename(columns={
    "precision": "pooled_precision_at_locked_threshold",
    "recall": "pooled_recall_at_locked_threshold",
    "f1": "pooled_f1_at_locked_threshold",
    "true_positives": "pooled_true_positives_at_locked_threshold",
    "false_positives": "pooled_false_positives_at_locked_threshold",
})
model_comparison_with_thresholds = mean_comparison.merge(
    pooled_operating_metrics, on="model_id", how="left", validate="one_to_one"
)

threshold_checks = {
    "Exactly six new pooled validation thresholds are selected":
        len(pooled_thresholds) == pooled_thresholds["model_id"].nunique() == 6,
    "Every pooled threshold is finite and bounded within [0,1]":
        np.isfinite(pooled_thresholds["threshold"]).all()
        and pooled_thresholds["threshold"].between(0, 1).all(),
    "Every pooled threshold is explicitly independent of Version 1":
        not pooled_thresholds["version_1_threshold_reused"].any()
        and pooled_thresholds["threshold_source"].str.contains(
            "pooled out-of-time validation", regex=False
        ).all(),
    "Exactly 18 fold-specific thresholds are diagnostic only":
        len(fold_threshold_stability) == 18
        and fold_threshold_stability["diagnostic_only"].all(),
    "The pooled locked threshold is applied unchanged to every fold and pooled scope":
        len(locked_threshold_performance) == 24
        and locked_threshold_performance.groupby("model_id")["operating_threshold"].nunique().eq(1).all(),
    "All recall, precision and F1 results are finite and bounded":
        np.isfinite(locked_threshold_performance[["recall", "precision", "f1"]].to_numpy()).all()
        and locked_threshold_performance[["recall", "precision", "f1"]].ge(0).all().all()
        and locked_threshold_performance[["recall", "precision", "f1"]].le(1).all().all(),
}
notebook_15_cell_04_gate = pd.DataFrame({
    "check": threshold_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in threshold_checks.values()],
    "passed": threshold_checks.values(),
})

pooled_thresholds.to_csv(
    NB15_OUTPUT_DIR / "v2_pooled_validation_f1_optimal_thresholds.csv", index=False
)
fold_threshold_stability.to_csv(
    NB15_OUTPUT_DIR / "v2_fold_specific_threshold_stability_diagnostic.csv", index=False
)
threshold_stability_summary.to_csv(
    NB15_OUTPUT_DIR / "v2_threshold_stability_summary.csv", index=False
)
locked_threshold_performance.to_csv(
    NB15_OUTPUT_DIR / "v2_locked_threshold_validation_performance.csv", index=False
)
model_comparison_with_thresholds.to_csv(
    NB15_OUTPUT_DIR / "v2_six_specification_final_validation_comparison.csv", index=False
)
notebook_15_cell_04_gate.to_csv(
    NB15_OUTPUT_DIR / "cell_04_threshold_selection_gate.csv", index=False
)

print("Notebook 15 Cell 4 validation gate:")
display(notebook_15_cell_04_gate)
print("\nNew pooled out-of-time validation F1 thresholds (one per specification):")
display(pooled_thresholds[[
    "model_family_display", "specification", "config_id", "threshold",
    "precision", "recall", "f1", "candidate_thresholds",
]])
print("\nFold-specific threshold stability diagnostic (not used downstream):")
display(threshold_stability_summary)
failed = [name for name, passed in threshold_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 4 blocked: " + "; ".join(failed))


In [ ]:
# Report quarterly top-risk capacity results within each fold and pooled validation

if not notebook_15_cell_04_gate["passed"].all():
    raise AssertionError("Cell 5 blocked because Cell 4 did not pass.")


def capacity_result(frame, capacity, model_id, scope):
    work = frame.copy()
    work["report_date"] = pd.to_datetime(work["report_date"]).dt.normalize()
    work["_cert_sort"] = pd.to_numeric(work["cert"], errors="raise").astype("int64")
    work["_idrssd_sort"] = pd.to_numeric(work["idrssd"], errors="raise").astype("int64")
    work = work.sort_values(
        ["report_date", "predicted_probability", "_cert_sort", "_idrssd_sort", "panel_row_id"],
        ascending=[True, False, True, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    work["rank_within_quarter"] = work.groupby("report_date", sort=False).cumcount() + 1
    work["active_bank_quarters_in_quarter"] = work.groupby("report_date", sort=False)[
        "panel_row_id"
    ].transform("size")
    work["flagged_group_size_in_quarter"] = np.maximum(
        1,
        np.ceil(capacity * work["active_bank_quarters_in_quarter"]).astype("int64"),
    )
    work["flagged"] = work["rank_within_quarter"] <= work["flagged_group_size_in_quarter"]

    quarterly = work.groupby("report_date", as_index=False).agg(
        active_bank_quarters=("panel_row_id", "size"),
        required_flagged_count=("flagged_group_size_in_quarter", "first"),
        realised_flagged_count=("flagged", "sum"),
    )
    if not quarterly["required_flagged_count"].eq(quarterly["realised_flagged_count"]).all():
        raise AssertionError(f"Quarterly flagged-count rule failed for {model_id}/{scope}/{capacity}.")

    positive = work[TARGET].eq(1)
    flagged = work["flagged"]
    failed_bank_key = work["cert"].astype("int64")
    distinct_failed_bank_denominator = int(failed_bank_key.loc[positive].nunique())
    distinct_failed_bank_numerator = int(failed_bank_key.loc[positive & flagged].nunique())
    positive_bank_quarter_denominator = int(positive.sum())
    positive_bank_quarter_numerator = int((positive & flagged).sum())
    flagged_total = int(flagged.sum())
    prevalence = positive_bank_quarter_denominator / len(work)
    precision = positive_bank_quarter_numerator / flagged_total
    family, specification = model_id.split("__", 1)

    return {
        "model_id": model_id,
        "model_family": family,
        "model_family_display": MODEL_INPUTS[family]["display_name"],
        "specification": specification,
        "scope": scope,
        "capacity": float(capacity),
        "capacity_label": CAPACITY_LABELS[capacity],
        "validation_quarters": int(work["report_date"].nunique()),
        "active_bank_quarters_denominator": int(len(work)),
        "flagged_bank_quarters_total": flagged_total,
        "flagged_bank_quarters_mean_per_validation_quarter": float(
            flagged_total / work["report_date"].nunique()
        ),
        "flagged_count_denominator_label":
            "eligible active bank-quarter rows ranked separately within each validation quarter",
        "distinct_failed_banks_captured": distinct_failed_bank_numerator,
        "distinct_failed_banks_denominator": distinct_failed_bank_denominator,
        "distinct_failed_bank_capture": float(
            distinct_failed_bank_numerator / distinct_failed_bank_denominator
        ),
        "distinct_failed_bank_denominator_label":
            "distinct CERTs with failed_1yr=1 in the stated validation scope",
        "distinct_failed_bank_capture_status": "PRIMARY",
        "positive_bank_quarters_captured": positive_bank_quarter_numerator,
        "positive_bank_quarters_denominator": positive_bank_quarter_denominator,
        "positive_bank_quarter_capture": float(
            positive_bank_quarter_numerator / positive_bank_quarter_denominator
        ),
        "positive_bank_quarter_denominator_label":
            "bank-quarter rows with failed_1yr=1 in the stated validation scope",
        "positive_bank_quarter_capture_status": "SECONDARY",
        "flagged_positive_bank_quarter_precision": float(precision),
        "positive_bank_quarter_prevalence": float(prevalence),
        "precision_divided_by_prevalence_lift": float(precision / prevalence),
        "capacity_affects_model_selection": False,
        "quarterly_group_size_rule": "max(1, ceil(capacity * active bank-quarter rows))",
        "ranking_tie_break_rule":
            "predicted_probability descending; CERT, IDRSSD, panel_row_id ascending",
    }, quarterly


capacity_rows = []
quarterly_flag_count_rows = []
for model_id, frame in sorted(predictions_by_model.items()):
    for scope in [*FOLDS, "pooled_oot_validation"]:
        scope_frame = frame if scope == "pooled_oot_validation" else frame.loc[
            frame["fold"].astype(str).eq(scope)
        ].copy()
        for capacity in CAPACITIES:
            result, quarterly = capacity_result(scope_frame, capacity, model_id, scope)
            capacity_rows.append(result)
            quarterly = quarterly.copy()
            quarterly.insert(0, "model_id", model_id)
            quarterly.insert(1, "scope", scope)
            quarterly.insert(2, "capacity", capacity)
            quarterly_flag_count_rows.append(quarterly)

capacity_results = pd.DataFrame(capacity_rows).sort_values(
    ["model_family", "specification", "scope", "capacity"]
).reset_index(drop=True)
quarterly_flag_count_audit = pd.concat(quarterly_flag_count_rows, ignore_index=True)

pooled_denominators = capacity_results.loc[
    capacity_results["scope"].eq("pooled_oot_validation")
].groupby("model_id", as_index=False).agg(
    active_bank_quarters_denominator=("active_bank_quarters_denominator", "first"),
    distinct_failed_banks_denominator=("distinct_failed_banks_denominator", "first"),
    positive_bank_quarters_denominator=("positive_bank_quarters_denominator", "first"),
    validation_quarters=("validation_quarters", "first"),
)

capacity_checks = {
    "All six models, four scopes and four capacities are reported":
        len(capacity_results) == 6 * 4 * 4
        and capacity_results.groupby(["model_id", "scope"])["capacity"].nunique().eq(4).all(),
    "Every quarterly flagged total follows the ceiling rule exactly":
        quarterly_flag_count_audit["required_flagged_count"].eq(
            quarterly_flag_count_audit["realised_flagged_count"]
        ).all()
        and quarterly_flag_count_audit["realised_flagged_count"].ge(1).all(),
    "Total flagged counts reconcile to mean per validation quarter":
        np.allclose(
            capacity_results["flagged_bank_quarters_total"]
            / capacity_results["validation_quarters"],
            capacity_results["flagged_bank_quarters_mean_per_validation_quarter"],
            rtol=0.0,
            atol=1e-12,
        ),
    "Primary and secondary capture denominators are positive and clearly labelled":
        capacity_results["distinct_failed_banks_denominator"].gt(0).all()
        and capacity_results["positive_bank_quarters_denominator"].gt(0).all()
        and capacity_results["distinct_failed_bank_capture_status"].eq("PRIMARY").all()
        and capacity_results["positive_bank_quarter_capture_status"].eq("SECONDARY").all()
        and capacity_results[
            [
                "flagged_count_denominator_label", "distinct_failed_bank_denominator_label",
                "positive_bank_quarter_denominator_label",
            ]
        ].notna().all().all(),
    "All capture, precision and lift results are finite and bounded where applicable":
        np.isfinite(capacity_results[
            [
                "distinct_failed_bank_capture", "positive_bank_quarter_capture",
                "flagged_positive_bank_quarter_precision", "precision_divided_by_prevalence_lift",
            ]
        ].to_numpy()).all()
        and capacity_results[
            [
                "distinct_failed_bank_capture", "positive_bank_quarter_capture",
                "flagged_positive_bank_quarter_precision",
            ]
        ].ge(0).all().all()
        and capacity_results[
            [
                "distinct_failed_bank_capture", "positive_bank_quarter_capture",
                "flagged_positive_bank_quarter_precision",
            ]
        ].le(1).all().all(),
    "All six models share identical pooled validation denominators":
        pooled_denominators.drop(columns="model_id").nunique().eq(1).all(),
    "Capacity results are explicitly excluded from model selection":
        not capacity_results["capacity_affects_model_selection"].any(),
}
notebook_15_cell_05_gate = pd.DataFrame({
    "check": capacity_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in capacity_checks.values()],
    "passed": capacity_checks.values(),
})

capacity_results.to_csv(
    NB15_OUTPUT_DIR / "v2_quarterly_top_risk_capacity_validation_results.csv", index=False
)
quarterly_flag_count_audit.to_csv(
    NB15_OUTPUT_DIR / "cell_05_quarterly_flag_count_audit.csv", index=False
)
pooled_denominators.to_csv(
    NB15_OUTPUT_DIR / "cell_05_pooled_capacity_denominators.csv", index=False
)
notebook_15_cell_05_gate.to_csv(
    NB15_OUTPUT_DIR / "cell_05_capacity_validation_gate.csv", index=False
)

print("Notebook 15 Cell 5 validation gate:")
display(notebook_15_cell_05_gate)
print("\nPooled out-of-time validation capacity results:")
display(capacity_results.loc[
    capacity_results["scope"].eq("pooled_oot_validation"),
    [
        "model_family_display", "specification", "capacity_label",
        "flagged_bank_quarters_total", "flagged_bank_quarters_mean_per_validation_quarter",
        "distinct_failed_banks_captured", "distinct_failed_banks_denominator",
        "distinct_failed_bank_capture", "positive_bank_quarters_captured",
        "positive_bank_quarters_denominator", "positive_bank_quarter_capture",
    ],
])
failed = [name for name, passed in capacity_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 5 blocked: " + "; ".join(failed))


In [ ]:
# Lock predictors, preprocessing, hyperparameters, thresholds and capacity definitions

if not notebook_15_cell_05_gate["passed"].all():
    raise AssertionError("Cell 6 blocked because Cell 5 did not pass.")

PREPROCESSING_LOCKS = {
    "logistic_regression": {
        "imputation": "training-sample median for every predictor",
        "winsorisation": "training-sample 1st and 99th percentiles for continuous predictors",
        "winsorisation_exclusion": [HISTORY_INDICATOR],
        "standardisation": "training-sample mean and population standard deviation for all predictors",
        "native_missing_value_treatment": "not used",
        "final_training_rule": "refit every preprocessing parameter on eligible final-training rows only",
        "fold_preprocessing_objects_reused_in_final_fit": False,
    },
    "random_forest": {
        "imputation": "training-sample median for every predictor",
        "winsorisation": "training-sample 1st and 99th percentiles for continuous predictors",
        "winsorisation_exclusion": [HISTORY_INDICATOR],
        "standardisation": "not used",
        "native_missing_value_treatment": "not used",
        "final_training_rule": "refit every preprocessing parameter on eligible final-training rows only",
        "fold_preprocessing_objects_reused_in_final_fit": False,
    },
    "xgboost": {
        "imputation": (
            "training-sample median for every predictor, including limited-history, genuine "
            "and FFIEC 051 structural missingness"
        ),
        "winsorisation": "training-sample 1st and 99th percentiles for continuous predictors",
        "winsorisation_exclusion": [HISTORY_INDICATOR],
        "standardisation": "not used",
        "native_missing_value_treatment": "prohibited",
        "final_training_rule": "refit every preprocessing parameter on eligible final-training rows only",
        "fold_preprocessing_objects_reused_in_final_fit": False,
    },
}

IMPLEMENTATION_LOCKS = {
    "logistic_regression": {
        "estimator": "sklearn.linear_model.LogisticRegression",
        "penalty": "l2", "solver": "lbfgs", "max_iter": 1000,
        "class_weight_rule": "{0: 1.0, 1: final-training negative/positive ratio * selected multiplier}",
        "random_seed": RANDOM_SEED,
    },
    "random_forest": {
        "estimator": "sklearn.ensemble.RandomForestClassifier",
        "criterion": "gini", "bootstrap": True, "max_samples": 0.50,
        "class_weight_rule": "{0: 1.0, 1: final-training negative/positive ratio * selected multiplier}",
        "random_seed": RANDOM_SEED, "n_jobs": -1,
    },
    "xgboost": {
        "estimator": "xgboost.XGBClassifier",
        "objective": "binary:logistic", "eval_metric": "logloss",
        "tree_method": "hist", "early_stopping_rounds": None,
        "scale_pos_weight_rule": "final-training negative/positive ratio * selected multiplier",
        "random_seed": RANDOM_SEED, "n_jobs": -1,
    },
}

threshold_lookup = pooled_thresholds.set_index("model_id")
comparison_lookup = mean_comparison.set_index("model_id")
locked_model_rows = []
locked_models_json = []

for model_id, selected_record in sorted(selected_records.items()):
    family, specification = model_id.split("__", 1)
    missing_hyperparameters = [
        name for name in MODEL_INPUTS[family]["hyperparameter_columns"]
        if name not in selected_record
    ]
    if missing_hyperparameters:
        raise KeyError(f"{model_id} selected handoff lacks: {missing_hyperparameters}")
    hyperparameters = {
        name: to_builtin(selected_record[name])
        for name in MODEL_INPUTS[family]["hyperparameter_columns"]
    }
    threshold_row = threshold_lookup.loc[model_id]
    comparison_row = comparison_lookup.loc[model_id]
    model_lock = {
        "model_id": model_id,
        "model_family": family,
        "model_family_display": MODEL_INPUTS[family]["display_name"],
        "specification": specification,
        "config_id": str(selected_record["config_id"]),
        "predictors_in_order": SPECIFICATION_FEATURES[specification],
        "predictor_count": len(SPECIFICATION_FEATURES[specification]),
        "preprocessing": PREPROCESSING_LOCKS[family],
        "missing_data_policy_from_shared_config": MISSING_DATA_POLICY,
        "hyperparameters": hyperparameters,
        "implementation": IMPLEMENTATION_LOCKS[family],
        "pooled_validation_f1_threshold": float(threshold_row["threshold"]),
        "threshold_provenance": str(threshold_row["threshold_source"]),
        "fold_specific_thresholds_used_downstream": False,
        "mean_validation_pr_auc": float(comparison_row["mean_pr_auc"]),
        "mean_validation_roc_auc": float(comparison_row["mean_roc_auc"]),
        "fold_pr_auc_sample_standard_deviation": float(comparison_row["std_pr_auc"]),
        "selection_status": str(comparison_row["selection_status"]),
    }
    locked_models_json.append(to_builtin(model_lock))
    locked_model_rows.append({
        "model_id": model_id,
        "model_family": family,
        "model_family_display": MODEL_INPUTS[family]["display_name"],
        "specification": specification,
        "config_id": str(selected_record["config_id"]),
        "predictor_count": len(SPECIFICATION_FEATURES[specification]),
        "predictors_in_order_json": json.dumps(SPECIFICATION_FEATURES[specification]),
        "preprocessing_json": json.dumps(to_builtin(PREPROCESSING_LOCKS[family]), sort_keys=True),
        "missing_data_policy_json": json.dumps(to_builtin(MISSING_DATA_POLICY), sort_keys=True),
        "hyperparameters_json": json.dumps(to_builtin(hyperparameters), sort_keys=True),
        "implementation_json": json.dumps(to_builtin(IMPLEMENTATION_LOCKS[family]), sort_keys=True),
        "pooled_validation_f1_threshold": float(threshold_row["threshold"]),
        "selection_status": str(comparison_row["selection_status"]),
        "final_test_used_for_selection": False,
        "version_1_threshold_reused": False,
    })

locked_model_specifications = pd.DataFrame(locked_model_rows)
source_artifacts = [
    {
        "file": str(path),
        "bytes": int(path.stat().st_size),
        "sha256": input_hashes_before[str(path)],
    }
    for path in loaded_input_paths
]

model_lock_manifest = {
    "schema_version": "v2_model_lock_manifest_1.0",
    "created_utc": RUN_UTC,
    "analysis_version": ANALYSIS_VERSION,
    "notebook": "15_Validation_Comparison_and_Model_Lock.ipynb",
    "timing_policy": TIMING_POLICY,
    "target_horizon_days": TARGET_HORIZON_DAYS,
    "validation_folds": VALIDATION_FOLDS,
    "selection": {
        "universe": "six Version 2 lagged-bank specifications",
        "primary_rule": "highest unweighted mean raw validation PR-AUC",
        "first_tie_break": "higher unweighted mean validation ROC-AUC within 1e-6",
        "second_tie_break": "lower sample standard deviation of fold PR-AUC within 1e-6 again",
        "tolerance": SELECTION_TOLERANCE,
        "leader_model_id": LEADER_MODEL_ID,
        "capacity_results_used": False,
        "threshold_results_used": False,
        "final_test_information_used": False,
    },
    "threshold": {
        "rule": "maximum F1 on pooled selected-model out-of-time validation predictions",
        "tie_breaks": ["higher recall", "higher precision", "higher threshold"],
        "one_threshold_per_specification": True,
        "fold_specific_thresholds_are_diagnostic_only": True,
        "version_1_threshold_reused": False,
        "cost_interpretation": (
            "F1 is a conventional descriptive balance between precision and recall, "
            "not a measure of supervisory costs."
        ),
    },
    "capacity": {
        "fractions": list(CAPACITIES),
        "ranking_unit": "active bank-quarter rows separately within each quarter",
        "group_size_rule": "max(1, ceil(capacity * active bank-quarter rows))",
        "ranking_tie_breaks": [
            "predicted_probability descending", "CERT ascending",
            "IDRSSD ascending", "panel_row_id ascending",
        ],
        "primary_result": "distinct-failed-bank capture",
        "primary_failed_bank_key": "CERT",
        "secondary_result": "positive-bank-quarter capture",
        "flagged_counts": ["total across scope", "mean per validation quarter"],
        "used_for_model_selection": False,
    },
    "locked_models": locked_models_json,
    "final_test_access_audit": {
        "final_test_files_loaded": 0,
        "final_test_rows_loaded": 0,
        "final_test_metrics_loaded": 0,
        "final_test_information_entered_selection": False,
    },
    "source_artifacts": source_artifacts,
}

MODEL_LOCK_MANIFEST_PATH = NB15_OUTPUT_DIR / "v2_model_lock_manifest.json"
with open(MODEL_LOCK_MANIFEST_PATH, "w", encoding="utf-8") as handle:
    json.dump(to_builtin(model_lock_manifest), handle, indent=2, sort_keys=True, allow_nan=False)
locked_model_specifications.to_csv(
    NB15_OUTPUT_DIR / "v2_locked_model_specifications.csv", index=False
)

capacity_definition = pd.DataFrame({
    "item": [
        "capacity fractions", "ranking unit", "group size", "score order",
        "tie-breaks", "primary result", "primary denominator",
        "secondary result", "secondary denominator", "flagged count reporting",
        "selection role",
    ],
    "locked_definition": [
        "1%, 5%, 10%, 20%", "active bank-quarter rows within each quarter",
        "max(1, ceil(capacity * active bank-quarter rows))",
        "descending predicted probability", "CERT, IDRSSD, panel_row_id ascending",
        "distinct-failed-bank capture",
        "distinct CERTs with failed_1yr=1 in the stated scope",
        "positive-bank-quarter capture",
        "bank-quarter rows with failed_1yr=1 in the stated scope",
        "total across scope and mean per validation quarter",
        "descriptive only; not used in model selection",
    ],
})
capacity_definition.to_csv(
    NB15_OUTPUT_DIR / "v2_locked_capacity_definition.csv", index=False
)

lock_checks = {
    "All six model specifications are locked":
        len(locked_model_specifications) == locked_model_specifications["model_id"].nunique() == 6,
    "Both predictor sets remain exact and include the history indicator":
        len(BANK_ONLY_PREDICTORS) == 19
        and len(BANK_PLUS_MACRO_PREDICTORS) == 23
        and all(HISTORY_INDICATOR in values for values in SPECIFICATION_FEATURES.values()),
    "Every model has complete tuned hyperparameters and implementation rules":
        locked_model_specifications[
            ["hyperparameters_json", "implementation_json", "preprocessing_json"]
        ].notna().all().all(),
    "Preprocessing is refitted on final training and fold objects are not reused":
        all(
            lock["final_training_rule"].startswith("refit")
            and lock["fold_preprocessing_objects_reused_in_final_fit"] is False
            for lock in PREPROCESSING_LOCKS.values()
        ),
    "Exactly one validation leader is recorded":
        locked_model_specifications["selection_status"].eq("LOCKED VALIDATION LEADER").sum() == 1
        and model_lock_manifest["selection"]["leader_model_id"] == LEADER_MODEL_ID,
    "Six pooled thresholds are locked with no Version 1 reuse":
        locked_model_specifications["pooled_validation_f1_threshold"].notna().all()
        and not locked_model_specifications["version_1_threshold_reused"].any(),
    "Capacity definitions are locked but excluded from selection":
        model_lock_manifest["capacity"]["fractions"] == list(CAPACITIES)
        and model_lock_manifest["capacity"]["used_for_model_selection"] is False,
    "The manifest confirms zero final-test access":
        model_lock_manifest["final_test_access_audit"] == {
            "final_test_files_loaded": 0,
            "final_test_rows_loaded": 0,
            "final_test_metrics_loaded": 0,
            "final_test_information_entered_selection": False,
        },
    "Machine-readable JSON manifest was written without NaN":
        MODEL_LOCK_MANIFEST_PATH.is_file()
        and "NaN" not in MODEL_LOCK_MANIFEST_PATH.read_text(encoding="utf-8"),
}
notebook_15_cell_06_gate = pd.DataFrame({
    "check": lock_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in lock_checks.values()],
    "passed": lock_checks.values(),
})
notebook_15_cell_06_gate.to_csv(
    NB15_OUTPUT_DIR / "cell_06_model_lock_gate.csv", index=False
)

print("Notebook 15 Cell 6 validation gate:")
display(notebook_15_cell_06_gate)
print("\nLocked model specifications:")
display(locked_model_specifications[[
    "model_family_display", "specification", "config_id", "predictor_count",
    "pooled_validation_f1_threshold", "selection_status",
]])
print("\nMachine-readable model lock:")
print(MODEL_LOCK_MANIFEST_PATH)
failed = [name for name, passed in lock_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 6 blocked: " + "; ".join(failed))


In [ ]:
# Prove input immutability and save the final Notebook 15 handoff

if not notebook_15_cell_06_gate["passed"].all():
    raise AssertionError("Cell 7 blocked because Cell 6 did not pass.")

input_hashes_after = {str(path): sha256_file(path) for path in loaded_input_paths}
input_immutability = pd.DataFrame([
    {
        "input_file": path,
        "sha256_before": input_hashes_before[path],
        "sha256_after": input_hashes_after[path],
        "unchanged": input_hashes_before[path] == input_hashes_after[path],
    }
    for path in input_hashes_before
])
input_immutability.to_csv(
    NB15_OUTPUT_DIR / "cell_07_input_immutability_audit.csv", index=False
)

leader_row = mean_comparison.loc[
    mean_comparison["model_id"].eq(LEADER_MODEL_ID)
].iloc[0]
notebook_15_final_summary = pd.DataFrame({
    "measure": [
        "Analysis version", "Compared specifications", "Validation folds",
        "Fold-level comparison rows", "Mean comparison rows", "Validation leader",
        "Leader mean raw PR-AUC", "Leader mean ROC-AUC", "Leader fold PR-AUC standard deviation",
        "New pooled F1 thresholds locked", "Fold-specific threshold diagnostics",
        "Quarterly capacity fractions", "Capacity result rows",
        "Primary capacity measure", "Secondary capacity measure",
        "Final-test files loaded", "Final-test rows loaded", "Final-test metrics loaded",
        "Version 1 threshold reused", "Input files modified", "Next notebook",
    ],
    "value": [
        ANALYSIS_VERSION, 6, 3, len(fold_comparison), len(mean_comparison), LEADER_MODEL_ID,
        float(leader_row["mean_pr_auc"]), float(leader_row["mean_roc_auc"]),
        float(leader_row["std_pr_auc"]), len(pooled_thresholds),
        len(fold_threshold_stability), str(CAPACITIES), len(capacity_results),
        "distinct-failed-bank capture", "positive-bank-quarter capture",
        0, 0, 0, False, int((~input_immutability["unchanged"]).sum()),
        "16_Final_Out_of_Time_Test.ipynb",
    ],
})
notebook_15_final_summary.to_csv(
    NB15_OUTPUT_DIR / "cell_07_final_summary.csv", index=False
)

earlier_gates = [
    notebook_15_cell_01_gate, notebook_15_cell_02_gate, notebook_15_cell_03_gate,
    notebook_15_cell_04_gate, notebook_15_cell_05_gate, notebook_15_cell_06_gate,
]
final_checks = {
    "All earlier Notebook 15 gates passed":
        all(gate["passed"].all() for gate in earlier_gates),
    "All six selected specifications were independently compared":
        len(mean_comparison) == 6 and len(fold_comparison) == 18,
    "One leader was selected using validation raw PR-AUC and locked tie-breakers":
        mean_comparison["selection_status"].eq("LOCKED VALIDATION LEADER").sum() == 1,
    "Six new pooled validation F1 thresholds were locked":
        len(pooled_thresholds) == 6
        and not pooled_thresholds["version_1_threshold_reused"].any(),
    "Fold-specific thresholds remain stability diagnostics only":
        len(fold_threshold_stability) == 18
        and fold_threshold_stability["diagnostic_only"].all(),
    "All fold and pooled quarterly capacity outputs were saved":
        len(capacity_results) == 96
        and set(capacity_results["capacity"].round(2)) == set(CAPACITIES),
    "Primary and secondary capacity denominators are explicit":
        capacity_results["distinct_failed_bank_capture_status"].eq("PRIMARY").all()
        and capacity_results["positive_bank_quarter_capture_status"].eq("SECONDARY").all(),
    "No final-test information entered model or threshold selection":
        model_lock_manifest["selection"]["final_test_information_used"] is False
        and model_lock_manifest["final_test_access_audit"]["final_test_rows_loaded"] == 0,
    "No Version 1 threshold was loaded or reused":
        model_lock_manifest["threshold"]["version_1_threshold_reused"] is False,
    "Every locked input remained byte-for-byte unchanged":
        input_immutability["unchanged"].all(),
}
notebook_15_cell_07_gate = pd.DataFrame({
    "check": final_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in final_checks.values()],
    "passed": final_checks.values(),
})
notebook_15_cell_07_gate.to_csv(
    NB15_OUTPUT_DIR / "cell_07_final_validation_gate.csv", index=False
)

principal_purposes = {
    "cell_01_setup_gate.csv": "V2 scope, timing and output-root gate",
    "cell_01_notebook_15_contract.csv": "Precommitted selection, threshold and capacity rules",
    "cell_02_model_handoff_audit.csv": "Notebook 12–14 selected-handoff checks",
    "cell_02_selected_prediction_source_manifest.csv": "Selected prediction source paths and hashes",
    "cell_02_preprocessing_handoff_audit.csv": "Training-only preprocessing handoff audit",
    "cell_02_input_hashes.csv": "Before-run hashes for every loaded input",
    "cell_02_handoff_validation_gate.csv": "Locked handoff validation gate",
    "cell_03_selected_prediction_audit.csv": "Selected prediction file and content audit",
    "cell_03_cross_model_row_reconciliation.csv": "Proof that all six models use identical validation rows",
    "v2_six_specification_fold_validation_comparison.csv": "Recalculated fold prevalence, PR-AUC, lift, ROC-AUC, recall and F1",
    "v2_six_specification_mean_validation_comparison.csv": "Unweighted three-fold comparison and leader",
    "cell_03_metric_recomputation_and_selection_gate.csv": "Metric and model-selection gate",
    "v2_pooled_validation_f1_optimal_thresholds.csv": "Six new pooled-validation F1 thresholds",
    "v2_fold_specific_threshold_stability_diagnostic.csv": "Diagnostic-only fold thresholds",
    "v2_threshold_stability_summary.csv": "Fold-threshold variation summary",
    "v2_locked_threshold_validation_performance.csv": "Recall, precision and F1 at locked pooled thresholds",
    "v2_six_specification_final_validation_comparison.csv": "Mean ranking metrics plus pooled locked-threshold results",
    "cell_04_threshold_selection_gate.csv": "Threshold provenance and completeness gate",
    "v2_quarterly_top_risk_capacity_validation_results.csv": "Primary and secondary capacity results for every fold and pooled validation",
    "cell_05_quarterly_flag_count_audit.csv": "Quarterly ceiling-rule flagged-count audit",
    "cell_05_pooled_capacity_denominators.csv": "Pooled denominator reconciliation across six models",
    "cell_05_capacity_validation_gate.csv": "Capacity definition and reconciliation gate",
    "v2_model_lock_manifest.json": "Machine-readable Version 2 model lock",
    "v2_locked_model_specifications.csv": "Flat six-specification model-lock table",
    "v2_locked_capacity_definition.csv": "Locked quarterly capacity definition",
    "cell_06_model_lock_gate.csv": "Predictor, preprocessing, hyperparameter and threshold lock gate",
    "cell_07_input_immutability_audit.csv": "Before-and-after hashes of every loaded input",
    "cell_07_final_summary.csv": "Notebook 15 final scope and result summary",
    "cell_07_final_validation_gate.csv": "Final fail-fast validation gate",
}
principal_paths = [NB15_OUTPUT_DIR / name for name in principal_purposes]
missing_outputs = [path for path in principal_paths if not path.is_file()]
if missing_outputs:
    raise AssertionError("Missing Notebook 15 outputs: " + "; ".join(map(str, missing_outputs)))

output_manifest = pd.DataFrame([
    {
        "file": str(path.relative_to(NB15_OUTPUT_DIR)),
        "purpose": principal_purposes[path.name],
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in principal_paths
])
output_manifest.to_csv(NB15_OUTPUT_DIR / "cell_07_output_manifest.csv", index=False)

print("Notebook 15 Cell 7 final validation gate:")
display(notebook_15_cell_07_gate)
print("\nNotebook 15 final summary:")
display(notebook_15_final_summary)
print("\nOutput manifest:")
display(output_manifest)
failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 7 blocked: " + "; ".join(failed))
if not (NB15_OUTPUT_DIR / "cell_07_output_manifest.csv").is_file():
    raise AssertionError("Notebook 15 output manifest was not written.")

print("\nNotebook 15 is PASS / MODEL LOCKED.")
print(f"Validation leader: {LEADER_MODEL_ID}")
print(f"Machine-readable model lock: {MODEL_LOCK_MANIFEST_PATH}")
print("Notebook 16 may now perform the one-time Version 2 final out-of-time evaluation.")
